# SDXL Convert Phase 2-5 (TPU v5e-8, 377GB RAM)

Consumes Phase 1 kernel's output (`data_sdxl.pkl` + `images_sdxl/`) from `/kaggle/input/...`.

Runs:
- Phase 2: `gen_quant_data_sdxl.py`
- Phase 3: `export_onnx_sdxl.py`
- Phase 4+5: `convert_all_sdxl.sh` (qairt converter + quantizer)

The QNN conversion uses CPU/RAM. A separate TPU-resident synthetic training burner keeps TPU work active during long CPU-only QNN stages and reports health every minute.


In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '__MIN_SOC__'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC} min_soc={MIN_SOC}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1
!grep -m1 'model name' /proc/cpuinfo
!grep -m1 'cpu MHz' /proc/cpuinfo
!lscpu | grep -E 'Model name|Socket|Thread|Core|Vendor|Flags' | head -10

In [ ]:
# Locate Phase 1 output
import glob, os
candidates = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert candidates, 'Phase 1 data_sdxl.pkl not found in /kaggle/input/ — did Phase 1 kernel complete?'
PHASE1_DIR = os.path.dirname(candidates[0])
print(f'Phase 1 dir: {PHASE1_DIR}')
!ls -lh $PHASE1_DIR

In [ ]:
import os
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME

In [ ]:
# Install tools + convertsdxl + shell helper
import os, subprocess

def _run(cmd, log=None, check=True):
    """bash with errexit + pipefail; pipe to tee if log. Raises on non-zero."""
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        wrapped = f'set -eo pipefail; {cmd} 2>&1 | tee {log}'
    else:
        wrapped = f'set -eo pipefail; {cmd}'
    rc = subprocess.call(['bash', '-c', wrapped])
    if check and rc != 0:
        raise RuntimeError(f'shell failed (rc={rc}): {cmd[:200]}  log={log}')
    return rc

# Ensure apt cache is fresh before any installs
_run('apt-get update -y > /dev/null')
_run('apt-get install -y unzip zip curl time > /dev/null')
# libc++/libc++abi/libunwind needed by QNN SDK libPyIrGraph.so / libQnnHtp.so — Kaggle Debian 13 trixie doesn't ship them
# Specific LLVM runtime packages — generic libc++-dev/libunwind-dev do NOT provide libunwind.so.1 (GNU libunwind ships .so.8)
_run('apt-get install -y libc++1-19 libc++abi1-19 libunwind-19 > /dev/null')
_run('ldconfig')  # refresh ld cache so newly installed libs are found by ldd / dlopen
_run('curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1')
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

_run('rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip')
_run("curl -sL --fail --retry 5 --retry-delay 5 -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'")
_run('mkdir -p /tmp/convertsdxl_unzip')
_run('unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip')

root = subprocess.check_output(
    "find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1",
    shell=True, text=True).strip()
assert root, 'export_sdxl.sh not found in unzipped convertsdxl'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
_run('mkdir -p /tmp/qnn_sdk')
_run(
    "curl -sL --fail --retry 5 --retry-delay 5 -A 'Mozilla/5.0' "
    "-o /tmp/qnn_sdk/qnn.zip "
    "'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'"
)
_run('cd /tmp/qnn_sdk && unzip -q qnn.zip')
bin_file = subprocess.check_output(
    'find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit',
    shell=True, text=True).strip()
assert bin_file, 'envsetup.sh not found in QNN SDK extract'
QNN_SDK_BIN = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN'] = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# Resolve model.safetensors from Phase 1 kernel_sources mount (read-only, no re-download)
import os, glob
candidates = glob.glob('/kaggle/input/**/model.safetensors', recursive=True)
assert candidates, 'model.safetensors not found in /kaggle/input/ — Phase 1 kernel_sources not mounted?'
os.environ['MODEL_PATH'] = candidates[0]
size = os.path.getsize(os.environ['MODEL_PATH'])
assert size > int(1e9), f'model.safetensors truncated: {size} bytes'
print(f'MODEL_PATH -> {os.environ["MODEL_PATH"]} ({size/1e9:.2f} GB)')

In [ ]:
# Python env
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
_run('uv venv -p 3.10.17 --clear')
_run('. .venv/bin/activate && uv sync')

In [ ]:
# Copy Phase 1 output into convert dir
import os, shutil, glob
pkl_list = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert pkl_list, 'data_sdxl.pkl not found in /kaggle/input/ — Phase 1 mount missing'
pkl = pkl_list[0]
p1_dir = os.path.dirname(pkl)
shutil.copy(pkl, f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
src_img = f'{p1_dir}/images_sdxl'
assert os.path.isdir(src_img), f'{src_img} missing — Phase 1 did not produce calibration images'
dst_img = f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'
if os.path.exists(dst_img):
    shutil.rmtree(dst_img)
shutil.copytree(src_img, dst_img)
assert len(os.listdir(dst_img)) > 0, f'{dst_img} empty after copy'
_run(f'ls -lh {os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
_run(f'ls {dst_img} | head -5')

In [ ]:
# Memory watcher + TPU synthetic training burner.
# The burner is intentionally not an LLM job: it trains a TPU-resident bf16 MLP with no Python-side data pipeline.
# This keeps real TPU forward/backward/optimizer work active while leaving CPU cores for QNN conversion.
import os, subprocess, time, json

os.makedirs('/kaggle/working/logs', exist_ok=True)

def _now():
    return time.strftime('%H:%M:%S')

# --- mem watcher (background, very light) ---
watcher_sh = '''#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
'''
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'], stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
open('/tmp/watcher.pid','w').write(str(p.pid))
print(f'[*] {_now()} mem-watcher pid={p.pid}')

# --- Install TF-TPU per Kaggle v5e TPU VM examples ---
# Keep this in system Python; QNN conversion uses its own uv venv.
# Do not set TF/OMP throttles in the notebook parent process: QNN inherits os.environ.
# The burner/watchdog pass those limits only to the TPU burner subprocess.
print(f'[*] {_now()} installing tensorflow-tpu runtime')
_run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax 2>&1 | tail -3', check=False)
_run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system --quiet '
     'tensorflow-tpu==2.18.0 '
     '--find-links https://storage.googleapis.com/libtpu-tf-releases/index.html')

burner_py = r'''
import os, time, json, traceback, signal, sys
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '1')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

LOG_DIR = '/kaggle/working/logs'
OK_FILE = f'{LOG_DIR}/tpu_burner.ok'
JSONL = f'{LOG_DIR}/tpu_burner.jsonl'
os.makedirs(LOG_DIR, exist_ok=True)

def log(msg):
    print('[tpu-burner ' + time.strftime('%H:%M:%S') + '] ' + str(msg), flush=True)

def write_ok(record):
    tmp = OK_FILE + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(record, f, sort_keys=True)
    os.replace(tmp, OK_FILE)
    with open(JSONL, 'a') as f:
        f.write(json.dumps(record, sort_keys=True) + '\n')

stopping = False
def _term(signum, frame):
    global stopping
    stopping = True
    log('signal received, exiting after current chunk')
signal.signal(signal.SIGTERM, _term)
signal.signal(signal.SIGINT, _term)

try:
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

    log('initializing TPU')
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    replicas = strategy.num_replicas_in_sync
    log('TPU ready replicas=' + str(replicas))

    DIM = 1024
    HIDDEN = 2048
    BATCH = 1024
    STEPS_PER_CHUNK = 8

    with strategy.scope():
        model = tf.keras.Sequential([
            tf.keras.layers.InputLayer(shape=(DIM,)),
            tf.keras.layers.Dense(HIDDEN, activation='gelu'),
            tf.keras.layers.Dense(HIDDEN, activation='gelu'),
            tf.keras.layers.Dense(DIM),
        ])
        opt = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-3)
        x = tf.ones((BATCH, DIM), dtype=tf.bfloat16)
        target = tf.zeros((BATCH, DIM), dtype=tf.bfloat16)
        _ = model(x, training=True)
        try:
            opt.build(model.trainable_variables)
        except AttributeError:
            pass

    @tf.function
    def train_chunk():
        total = tf.constant(0.0, tf.float32)
        for _ in range(STEPS_PER_CHUNK):
            with tf.GradientTape() as tape:
                pred = model(x, training=True)
                diff = tf.cast(pred - target, tf.float32)
                loss = tf.reduce_mean(diff * diff)
            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))
            total += loss
        return total / tf.cast(STEPS_PER_CHUNK, tf.float32)

    chunk = 0
    last_log = 0.0
    while not stopping:
        t0 = time.time()
        per_replica = strategy.run(train_chunk)
        reduced = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)
        loss = float(reduced.numpy())
        elapsed = time.time() - t0
        chunk += 1
        record = {
            'epoch': int(time.time()),
            'datetime': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'chunk': chunk,
            'replicas': int(replicas),
            'steps_per_chunk': STEPS_PER_CHUNK,
            'elapsed_sec': round(elapsed, 3),
            'loss': loss,
        }
        write_ok(record)
        now = time.time()
        if now - last_log >= 30:
            log('chunk={chunk} loss={loss:.6f} elapsed={elapsed:.2f}s replicas={replicas}'.format(
                chunk=chunk, loss=loss, elapsed=elapsed, replicas=replicas))
            last_log = now
        time.sleep(0.2)
    log('clean exit')
except Exception as e:
    log('FATAL ' + repr(e))
    traceback.print_exc()
    sys.exit(2)
'''
open('/kaggle/working/logs/tpu_burner.py','w').write(burner_py)

watchdog_py = r'''
import os, time, subprocess, signal
LOG_DIR = '/kaggle/working/logs'
BURNER = f'{LOG_DIR}/tpu_burner.py'
OK_FILE = f'{LOG_DIR}/tpu_burner.ok'
PID_FILE = '/tmp/tpu_burner.pid'
BURNER_LOG = f'{LOG_DIR}/tpu_burner.log'
WATCHDOG_LOG = f'{LOG_DIR}/tpu_watchdog.log'
PY = '/usr/local/bin/python3'
STALE_SEC = 180
FIRST_OK_STALE_SEC = 600
CHECK_SEC = 60

def log(msg):
    line = '[tpu-watchdog ' + time.strftime('%H:%M:%S') + '] ' + str(msg)
    print(line, flush=True)
    with open(WATCHDOG_LOG, 'a') as f:
        f.write(line + '\n')

def read_pid():
    try:
        return int(open(PID_FILE).read().strip())
    except Exception:
        return None

def pid_alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0)
        return True
    except ProcessLookupError:
        return False
    except PermissionError:
        return True

def stop_pid(pid):
    if not pid_alive(pid):
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    time.sleep(10)
    if pid_alive(pid):
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

def start_burner(reason):
    old = read_pid()
    if old:
        stop_pid(old)
    env = os.environ.copy()
    env.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
    env.setdefault('TF_NUM_INTRAOP_THREADS', '1')
    env.setdefault('TF_NUM_INTEROP_THREADS', '1')
    env.setdefault('OMP_NUM_THREADS', '1')
    logf = open(BURNER_LOG, 'ab', buffering=0)
    proc = subprocess.Popen([PY, BURNER], stdout=logf, stderr=subprocess.STDOUT, env=env, cwd='/kaggle/working')
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    log('started burner pid={} reason={}'.format(proc.pid, reason))

start_burner('initial')
while True:
    time.sleep(CHECK_SEC)
    pid = read_pid()
    age = None
    if os.path.exists(OK_FILE):
        age = time.time() - os.path.getmtime(OK_FILE)
    if not pid_alive(pid):
        start_burner('pid_dead')
    elif age is None:
        log('waiting for first ok pid={}'.format(pid))
        if os.path.exists(BURNER_LOG) and os.path.getsize(BURNER_LOG) > 0 and time.time() - os.path.getmtime(BURNER_LOG) > FIRST_OK_STALE_SEC:
            start_burner('no_ok_stale_log')
    elif age > STALE_SEC:
        start_burner('ok_stale_{:.0f}s'.format(age))
    else:
        log('healthy pid={} ok_age={:.0f}s'.format(pid, age))
'''
open('/kaggle/working/logs/tpu_watchdog.py','w').write(watchdog_py)

wd_log = open('/kaggle/working/logs/tpu_watchdog.stdout','ab', buffering=0)
wd = subprocess.Popen(['/usr/local/bin/python3', '/kaggle/working/logs/tpu_watchdog.py'],
                      stdout=wd_log, stderr=subprocess.STDOUT, cwd='/kaggle/working')
open('/tmp/tpu_watchdog.pid','w').write(str(wd.pid))
print(f'[*] {_now()} tpu-watchdog pid={wd.pid}')

# Require at least two successful TPU training chunks before entering long QNN work.
deadline = time.time() + 600
seen_chunks = set()
while time.time() < deadline:
    if wd.poll() is not None:
        raise RuntimeError('TPU watchdog died early')
    if os.path.exists('/kaggle/working/logs/tpu_burner.ok'):
        try:
            rec = json.load(open('/kaggle/working/logs/tpu_burner.ok'))
            seen_chunks.add(int(rec.get('chunk', 0)))
            if len(seen_chunks) >= 2:
                print(f'[*] {_now()} TPU burner verified chunks={sorted(seen_chunks)[-2:]} elapsed={rec.get("elapsed_sec")}s loss={rec.get("loss")}')
                _run('tail -20 /kaggle/working/logs/tpu_burner.log', check=False)
                _run('tail -10 /kaggle/working/logs/tpu_watchdog.log', check=False)
                break
        except Exception as e:
            print(f'[*] {_now()} waiting for TPU burner ok parse: {e}')
    time.sleep(10)
else:
    _run('tail -80 /kaggle/working/logs/tpu_burner.log', check=False)
    _run('tail -80 /kaggle/working/logs/tpu_watchdog.log', check=False)
    raise RuntimeError('TPU burner did not produce two successful chunks within 10 minutes')



In [ ]:
# Phase 2
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py',
    log='/kaggle/working/logs/phase2.log'
)
print(f'Phase 2 elapsed: {int(time.time()-t0)}s')
_run('free -h')

In [ ]:
# Phase 3
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py --model_path "$MODEL_PATH"',
    log='/kaggle/working/logs/phase3.log'
)
print(f'Phase 3 elapsed: {int(time.time()-t0)}s')
_run('free -h')
_run(r'find . -name "*.onnx" -exec ls -lh {} \;')
_run(r'find . -name "weights.pb" -exec ls -lh {} \;')

In [ ]:
# Backup Phase 3 ONNX output to /kaggle/working/phase3_backup
import os, shutil
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp): shutil.rmtree(bkp)
os.makedirs(bkp)
expected = ['clip_sdxl','clip2_sdxl','unet_sdxl','vae_decoder_sdxl','vae_encoder_sdxl']
missing = [d for d in expected if not os.path.exists(f'{os.environ["NPUCONVERT_DIR"]}/{d}')]
assert not missing, f'Phase 3 output dirs missing (partial export?): {missing}'
for d_ in expected:
    shutil.copytree(f'{os.environ["NPUCONVERT_DIR"]}/{d_}', f'{bkp}/{d_}')
_run(f'ls {bkp}')
_run(f'du -sh {bkp}/*')
_run(rf'find {os.environ["NPUCONVERT_DIR"]} -maxdepth 2 -name "input_list_*.txt" -exec cp {{}} {bkp}/ \;')
_run(f'ls {bkp}/*.txt', check=False)  # .txt copy is cosmetic

In [ ]:
# Patch convert scripts: replace hardcoded QNN_SDK_ROOT, make all config paths ABSOLUTE to NPUCONVERT_DIR.
# Reason: qnn-context-binary-generator's --config_file ./x.json is cwd-relative; if cwd drifts between
# qairt-quantizer (22min) and qnn-context-binary-generator we fail with "Cannot open file".
# Absolute paths eliminate the entire class of problems.
import os, re, glob, json as _json

NPU = os.environ['NPUCONVERT_DIR']
QNN = os.environ['QNN_SDK_ROOT_DIR']
SOC = os.environ['MIN_SOC']  # e.g. '8gen3' or '8gen4' (upstream default moved 8gen4 -> 8gen3 on 2026-04-20)

# 1. convert_all_sdxl.sh: replace hardcoded QNN_SDK_ROOT + prepend cd to NPUCONVERT_DIR
main = f'{NPU}/scripts/convert_all_sdxl.sh'
orig = open(main).read()
pat = re.compile(r'QNN_SDK_ROOT=/data/qairt/[0-9.]+')
assert pat.search(orig), f'QNN_SDK_ROOT pattern not found in {main}'
patched = pat.sub(f'QNN_SDK_ROOT={QNN}', orig)
patched = patched.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
assert patched != orig
open(main, 'w').write(patched)

# 2. Each sub-script: prepend defensive cd
for sub in glob.glob(f'{NPU}/scripts/convert_*.sh'):
    if sub.endswith('convert_all_sdxl.sh'):
        continue
    s = open(sub).read()
    s2 = s.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
    open(sub, 'w').write(s2)
    print(f'patched {os.path.basename(sub)}: prepended cd')

# 3. Rewrite htp_backend_<SOC>.json so config_file_path is absolute
hbp = f'{NPU}/htp_backend_{SOC}.json'
assert os.path.exists(hbp), f'{hbp} missing — upstream convertsdxl.zip may not ship this SOC config'
d = _json.load(open(hbp))
d['backend_extensions']['config_file_path'] = f'{NPU}/htp_config_{SOC}.json'
_json.dump(d, open(hbp, 'w'), indent=2)
print(f'htp_backend_{SOC}.json now:', open(hbp).read())

# 4. Sanity grep
_run(f'grep -nE "QNN_SDK_ROOT|^cd |source envsetup" {main}')

In [ ]:
# Phase 4-5 - QNN convert + quantize under a supervisor.
# The supervisor keeps notebook output active every minute, filters QNN flood, and reports TPU burner health.
import time, os, subprocess, select, json

t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
SOC = os.environ['MIN_SOC']

# qairt-converter -> libPyIrGraph.so -> libpython3.10.so.1.0 (NEEDED).
_pylib = subprocess.check_output(
    ['.venv/bin/python', '-c', 'import sys,os; print(os.path.join(sys.base_prefix, "lib"))'],
    text=True).strip()
assert os.path.exists(os.path.join(_pylib, 'libpython3.10.so.1.0')), \
    f'libpython3.10.so.1.0 not in {_pylib}'
os.environ['LD_LIBRARY_PATH'] = f"{_pylib}:{os.environ.get('LD_LIBRARY_PATH','')}"
print(f'[*] {time.strftime("%H:%M:%S")} libpython path ready: {_pylib}')

# Pre-flight: every critical QNN .so must have ldd resolve all deps.
_qnn = os.environ['QNN_SDK_ROOT_DIR']
_qnn_lib = f'{_qnn}/lib/x86_64-linux-clang'
_check_env = os.environ.copy()
_check_env['LD_LIBRARY_PATH'] = f'{_qnn_lib}:{_check_env["LD_LIBRARY_PATH"]}'
_critical_libs = [
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrGraph.so',
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrQuantizer.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnHtp.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnModelDlc.so',
]
for _lib in _critical_libs:
    assert os.path.exists(_lib), f'{_lib} not found in SDK extract'
    _ldd = subprocess.check_output(['ldd', _lib], env=_check_env, text=True)
    _missing = [l.strip() for l in _ldd.splitlines() if 'not found' in l]
    if _missing:
        print(f'FULL ldd output for {_lib}:\n{_ldd}')
        raise AssertionError(f'{_lib} unresolved deps:\n' + '\n'.join(_missing))
    print(f'[*] {time.strftime("%H:%M:%S")} ldd OK {os.path.basename(_lib)}')

# Pre-flight: config + input files must exist right before kick-off.
_npu = os.environ['NPUCONVERT_DIR']
_required = [
    f'{_npu}/htp_backend_{SOC}.json',
    f'{_npu}/htp_config_{SOC}.json',
    f'{_npu}/tokenizer.json',
    f'{_npu}/MNNConvert',
    f'{_npu}/scripts/convert_all_sdxl.sh',
    f'{_npu}/data_sdxl.pkl',
    f'{_npu}/vae_encoder_sdxl/model.onnx',
    f'{_npu}/vae_decoder_sdxl/model.onnx',
    f'{_npu}/unet_sdxl/model.onnx',
    f'{_npu}/clip_sdxl/model.onnx',
    f'{_npu}/clip2_sdxl/model.onnx',
    f'{_npu}/input_list_unet_sdxl.txt',
    f'{_npu}/input_list_vae_decoder_sdxl.txt',
    f'{_npu}/input_list_vae_encoder_sdxl.txt',
]
_missing_files = [p for p in _required if not os.path.exists(p)]
assert not _missing_files, 'Required files missing at Phase 4-5 kick-off:\n' + '\n'.join(_missing_files)
print(f'[*] {time.strftime("%H:%M:%S")} pre-flight OK files={len(_required)}')

os.makedirs('/kaggle/working/logs', exist_ok=True)
phase_log = '/kaggle/working/logs/phase45.log'
open(phase_log, 'w').close()

PRINT_KEYWORDS = [
    '===', 'INFO_CONVERSION_SUCCESS', 'Quantization completed successfully',
    'Quantized Model saved at', 'INFO_WRITE_SUCCESS', 'Converted Success!',
    'Model larger than 2GB', 'Elapsed (wall clock)', 'Maximum resident set size',
    'ERROR', 'Error', 'error:', 'Traceback', 'Cannot open', 'not found', 'No such file',
    'Killed', 'Aborted', 'Segmentation fault', 'context-binary', 'qnn-context-binary-generator',
    'qairt-quantizer', 'qairt-converter'
]
DROP_KEYWORDS = ['Failed to set thread affinity for cpuset']

last_milestone = 'starting'
suppressed_affinity = 0
printed_lines = 0

def mem_avail_mb():
    try:
        for line in open('/proc/meminfo'):
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) // 1024
    except Exception:
        pass
    return -1

def file_mib(path):
    try:
        return os.path.getsize(path) / (1024 * 1024)
    except OSError:
        return 0.0

def tpu_ok_age():
    p = '/kaggle/working/logs/tpu_burner.ok'
    if not os.path.exists(p):
        return None, None
    age = time.time() - os.path.getmtime(p)
    try:
        rec = json.load(open(p))
    except Exception:
        rec = None
    return age, rec

def tail_line(path):
    try:
        with open(path, 'rb') as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - 4096))
            lines = f.read().decode('utf-8', 'replace').splitlines()
            return lines[-1] if lines else ''
    except Exception:
        return ''

def is_dropped(line):
    return any(k in line for k in DROP_KEYWORDS)

def should_print(line):
    return any(k in line for k in PRINT_KEYWORDS)

def handle_line(line, logf):
    global last_milestone, printed_lines, suppressed_affinity
    logf.write(line)
    logf.flush()
    clean = line.rstrip()
    if is_dropped(clean):
        suppressed_affinity += 1
        return
    if clean:
        last_milestone = clean[-220:]
    if should_print(clean):
        print(clean[:1000], flush=True)
        printed_lines += 1

cmd = f'cd "{_npu}" && . .venv/bin/activate && /usr/bin/time -v bash scripts/convert_all_sdxl.sh --min_soc "$MIN_SOC"'
print(f'[*] {time.strftime("%H:%M:%S")} convert_all_sdxl.sh start soc={SOC}')
qnn_env = os.environ.copy()
for _k in ['TF_NUM_INTRAOP_THREADS', 'TF_NUM_INTEROP_THREADS', 'OMP_NUM_THREADS']:
    qnn_env.pop(_k, None)
proc = subprocess.Popen(['bash', '-lc', 'set -o pipefail; ' + cmd],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, cwd=_npu, env=qnn_env)

last_status = 0.0
with open(phase_log, 'a', buffering=1) as logf:
    while True:
        ready, _, _ = select.select([proc.stdout], [], [], 1)
        if ready:
            line = proc.stdout.readline()
            if line:
                handle_line(line, logf)
            elif proc.poll() is not None:
                break
        if proc.poll() is not None:
            for line in proc.stdout:
                handle_line(line, logf)
            break
        now = time.time()
        if now - last_status >= 60:
            age, rec = tpu_ok_age()
            if age is None:
                tpu_status = 'missing'
            else:
                tpu_status = f'age={int(age)}s chunk={rec.get("chunk") if rec else "?"} elapsed={rec.get("elapsed_sec") if rec else "?"}'
            wd_tail = tail_line('/kaggle/working/logs/tpu_watchdog.log')[-180:]
            print('[*] {ts} QNN running elapsed={elapsed}s log={log:.1f}MiB mem_avail={mem}MiB tpu_ok={tpu} suppressed_affinity={sup} last="{last}" watchdog="{wd}"'.format(
                ts=time.strftime('%H:%M:%S'), elapsed=int(now - t0), log=file_mib(phase_log),
                mem=mem_avail_mb(), tpu=tpu_status, sup=suppressed_affinity,
                last=last_milestone[-180:].replace('"', "'"), wd=wd_tail.replace('"', "'")))
            last_status = now

rc = proc.wait()
print(f'[*] {time.strftime("%H:%M:%S")} convert_all_sdxl.sh exited rc={rc} elapsed={int(time.time()-t0)}s printed_lines={printed_lines} suppressed_affinity={suppressed_affinity}')
_run('free -h')
for p in [f'{_npu}/htp_backend_{SOC}.json', f'{_npu}/htp_config_{SOC}.json']:
    print(f'post-run exists({p}): {os.path.exists(p)}')
_run(r'echo -n "phase45 log bytes: "; wc -c /kaggle/working/logs/phase45.log || true', check=False)
_run(r'echo -n "suppressed thread-affinity lines: "; grep -c "thread affinity for cpuset" /kaggle/working/logs/phase45.log || echo 0', check=False)
if rc != 0:
    _run('tail -120 /kaggle/working/logs/phase45.log', check=False)
    _run('tail -80 /kaggle/working/logs/tpu_burner.log', check=False)
    _run('tail -80 /kaggle/working/logs/tpu_watchdog.log', check=False)
    raise RuntimeError(f'convert_all_sdxl.sh failed rc={rc}')


In [ ]:
# Package output zip and stop background helpers.
import os, shutil, signal, time

def _kill_pidfile(pidfile, name):
    try:
        pid = int(open(pidfile).read().strip())
    except Exception:
        print(f'{name} stop skipped: no pidfile')
        return
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'[*] {time.strftime("%H:%M:%S")} SIGTERM {name} pid={pid}')
    except ProcessLookupError:
        print(f'{name} already stopped')
        return
    time.sleep(5)
    try:
        os.kill(pid, signal.SIGKILL)
        print(f'[*] {time.strftime("%H:%M:%S")} SIGKILL {name} pid={pid}')
    except ProcessLookupError:
        pass

os.chdir(os.environ['NPUCONVERT_DIR'])
out_dir = f'output/qnn_models_sdxl_{os.environ["MIN_SOC"]}'
assert os.path.isdir(out_dir), f'{out_dir} missing - Phase 4-5 produced no output'

# Free /kaggle/working space: Phase 4-5 succeeded, so phase3_backup is no longer needed.
# /kaggle/working has a 20GB cap; keeping backup + zip can exceed it.
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp):
    shutil.rmtree(bkp)
    print('reclaimed phase3_backup space')
_run('df -h /kaggle/working')

_run(f'touch {out_dir}/SDXL')
zip_name = f'{os.environ["MODEL_NAME"]}_qnn2.28_{os.environ["MIN_SOC"]}.zip'
_run(f'zip -r /kaggle/working/{zip_name} {out_dir} > /kaggle/working/logs/package_zip.log')
_run(f'ls -lh /kaggle/working/{zip_name}')
_run('df -h /kaggle/working')

# Stop TPU burner after packaging succeeds.
_kill_pidfile('/tmp/tpu_watchdog.pid', 'tpu-watchdog')
_kill_pidfile('/tmp/tpu_burner.pid', 'tpu-burner')
_kill_pidfile('/tmp/watcher.pid', 'mem-watcher')


In [ ]:
# Summary
import pandas as pd, os, time
print(f'[*] {time.strftime("%H:%M:%S")} summary start')
if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'mem samples: {len(df)}')
    if len(df):
        print('5 lowest MemAvail (MB):')
        print(df.nsmallest(5,'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
        print('5 highest TopRSS (MB):')
        print(df.nlargest(5,'TopRSS_MB')[['datetime','TopRSS_MB','TopProc']].to_string())
if os.path.exists('/kaggle/working/logs/tpu_burner.jsonl'):
    lines = open('/kaggle/working/logs/tpu_burner.jsonl').read().splitlines()
    print(f'tpu burner records: {len(lines)}')
    if lines:
        print('last tpu burner record:', lines[-1][:500])
_run('ls -lh /kaggle/working/', check=False)
_run('ls -lh /kaggle/working/logs/', check=False)
for log in ['tpu_burner.log', 'tpu_watchdog.log', 'phase45.log', 'package_zip.log']:
    p = f'/kaggle/working/logs/{log}'
    if os.path.exists(p):
        print(f'--- tail {log} ---')
        _run(f'tail -40 {p}', check=False)
